# Medicinal chemistry filter — STAT3 inhibitor screening

Companion tool for the study **"Screening STAT3 inhibitors from cucurbitacin E
derivatives using a medicinal chemistry filter and molecular docking"**.

Evaluates the drug-likeness of a **single compound** or a **CSV file of compounds**
against the eight criteria used in the study:

| Type | Criterion | Threshold |
|---|---|---|
| Rule | Lipinski (MW, LogP, HBA, HBD) | at most 1 of 4 violated |
| Rule | Pfizer 3/75 | not both LogP ≥ 3 and TPSA ≤ 75 |
| Measure | Fsp3 | ≥ 0.42 |
| Measure | MCE-18 | ≥ 45 |
| Measure | SA score | ≤ 6 |
| Alert | PAINS | no substructure match |
| Alert | BMS | no substructure match |
| Alert | Chelator | no substructure match |

> Lipinski's MW cutoff is relaxed from 500 to **600 g/mol**. NP score is reported
> but **not** used to reject compounds, since all derivatives are semi-synthesised
> from cucurbitacin E.

---
### How to use
1. Run **1. Input Package** once per session.
2. Run **2. Input Database** to load the study's compound libraries (optional).
3. Use **3A** for a single compound, or **3B** for a CSV file of compounds.


## 1. Input Package

In [ ]:
#@title Install and import libraries { display-mode: "form" }
#@markdown Click ▶ on the left. The first run takes about 1–2 minutes.

!pip install -q rdkit pandas tqdm
!pip install -q scopy

import os, sys, warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from tqdm.auto import tqdm

from rdkit import Chem, RDLogger
from rdkit.Chem import RDConfig
from rdkit.Chem.rdMolDescriptors import CalcNumHBA, CalcNumHBD, CalcFractionCSP3, CalcTPSA
from rdkit.Chem.Descriptors import MolWt
from rdkit.Chem.Crippen import MolLogP
RDLogger.DisableLog("rdApp.*")

# SA score and NP score live in the RDKit Contrib directory
sys.path.append(os.path.join(RDConfig.RDContribDir, "SA_Score"))
sys.path.append(os.path.join(RDConfig.RDContribDir, "NP_Score"))
import sascorer
import npscorer
_fscore = npscorer.readNPModel()

print("RDKit ready.")


In [ ]:
#@title Define descriptors and structural alerts { display-mode: "form" }
#@markdown Formulas are kept exactly as used in the study.

# ---- 1. Descriptors ----------------------------------------------------
def calculate_mce18(smiles):
    """MCE-18 as implemented in the study."""
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    num_atoms = mol.GetNumAtoms()
    num_bonds = mol.GetNumBonds()
    num_rings = mol.GetRingInfo().NumRings()
    num_H     = sum(atom.GetTotalNumHs() for atom in mol.GetAtoms())
    return num_atoms + num_bonds + 2 * num_rings - num_H

def calculate_np_score(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    try:
        return npscorer.scoreMol(mol, _fscore)
    except Exception:
        return None

def compute_descriptors(smiles):
    """Return the nine descriptors, or None if the SMILES is invalid."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return None
    return {
        "MW":       MolWt(mol),
        "LogP":     MolLogP(mol),
        "HBA":      CalcNumHBA(mol),
        "HBD":      CalcNumHBD(mol),
        "TPSA":     CalcTPSA(mol),
        "Fsp3":     CalcFractionCSP3(mol),
        "SA":       sascorer.calculateScore(mol),
        "MCE18":    calculate_mce18(smiles),
        "NP_Score": calculate_np_score(smiles),
    }

# ---- 2. PAINS / BMS / Chelator substructure alerts ---------------------
_SCOPY_OK = False
try:
    from scopy.ScoFH import fh_filter
    _SCOPY_OK = True
except Exception:
    from rdkit.Chem import FilterCatalog
    _params = FilterCatalog.FilterCatalogParams()
    _params.AddCatalog(FilterCatalog.FilterCatalogParams.FilterCatalogs.PAINS)
    _pains_cat = FilterCatalog.FilterCatalog(_params)

def structural_alerts(smiles):
    """True = no substructure match (compound accepted)."""
    mol = Chem.MolFromSmiles(str(smiles))
    if mol is None:
        return {"PAINS": None, "BMS": None, "CHE": None}
    if _SCOPY_OK:
        return {
            "PAINS": fh_filter.Check_PAINS(mol, detail=False)["Disposed"] == "Accepted",
            "BMS":   fh_filter.Check_BMS(mol,   detail=False)["Disposed"] == "Accepted",
            "CHE":   fh_filter.Check_Chelating(mol, detail=False)["Disposed"] == "Accepted",
        }
    # Fallback: PAINS only, via the RDKit FilterCatalog
    return {"PAINS": not _pains_cat.HasMatch(mol), "BMS": None, "CHE": None}

# ---- 3. Thresholds ------------------------------------------------------
def lipinski_violations(d):
    return int(d["HBA"] > 10) + int(d["HBD"] > 5) + int(d["MW"] > 600) + int(d["LogP"] > 5)

def evaluate(smiles):
    """Descriptors + per-criterion outcome + overall verdict."""
    d = compute_descriptors(smiles)
    if d is None:
        return None
    a = structural_alerts(smiles)
    nvio = lipinski_violations(d)
    res = dict(d)
    res["Lipinski_violations"] = nvio
    res["Lipinski_Pass"] = nvio <= 1
    res["Pfizer_pass"]   = not (d["LogP"] >= 3 and d["TPSA"] <= 75)
    res["Fsp3_pass"]     = d["Fsp3"]  >= 0.42
    res["MCE_pass"]      = d["MCE18"] >= 45
    res["SA_pass"]       = d["SA"]    <= 6
    res["NP_pass"]       = -5 <= d["NP_Score"] <= 5 if d["NP_Score"] is not None else None
    res.update(a)
    checks = [res["Lipinski_Pass"], res["Pfizer_pass"], res["Fsp3_pass"],
              res["MCE_pass"], res["SA_pass"], res["PAINS"], res["BMS"], res["CHE"]]
    res["Criteria_passed"] = sum(1 for c in checks if c is True)
    res["Final_pass"] = all(c is True for c in checks)
    return res

CRITERIA = [("Lipinski_Pass", "Lipinski (max 1 violation)"),
            ("Pfizer_pass",   "Pfizer 3/75"),
            ("Fsp3_pass",     "Fsp3 >= 0.42"),
            ("MCE_pass",      "MCE-18 >= 45"),
            ("SA_pass",       "SA score <= 6"),
            ("PAINS",         "PAINS alert"),
            ("BMS",           "BMS alert"),
            ("CHE",           "Chelator alert")]

print("Functions loaded. Scopy:", "yes" if _SCOPY_OK else "NO (PAINS only)")


## 2. Input Database

In [ ]:
#@title Load the study's compound libraries { display-mode: "form" }
#@markdown The in-house library of 1056 cucurbitacin E derivatives and the 85
#@markdown compounds that passed the filter. Skip this if you only want to
#@markdown screen your own compounds.

RAW = "https://raw.githubusercontent.com/phuocchung123/stat3/main"
df_1056 = pd.read_csv(f"{RAW}/data/1056_compounds.csv")
df_85   = pd.read_csv(f"{RAW}/data/85_compounds.csv",  index_col=0)
df_22   = pd.read_csv(f"{RAW}/data/22_compounds.csv")

print(f"In-house library      : {len(df_1056):>5} compounds")
print(f"Passed the filter     : {len(df_85):>5} compounds")
print(f"Active set (retrosp.) : {len(df_22):>5} compounds")
df_1056.head(3)


## 3. Screening

### 3A. Single compound

In [ ]:
#@title Enter the compound to screen { display-mode: "form" }
#@markdown Enter a SMILES string and any identifier, then click ▶.
#@markdown The default is the first of the 85 compounds that passed the filter.

query_SMILES = "CC(=O)OC(C)(C)/C=C/[C@@H](NCC(=O)O)[C@](C)(O)[C@H]1C(=O)C[C@@]2(C)[C@@H]3CC=C4[C@@H](C=C(O)C(=O)C4(C)C)[C@]3(C)[C@@H](O)C[C@]12C"  #@param {type:"string"}
query_ID     = "Compound_01"  #@param {type:"string"}

r = evaluate(query_SMILES)
if r is None:
    print("Invalid SMILES, please check the input.")
else:
    print(f"=== {query_ID} ===\n")
    print("DESCRIPTORS")
    for k, lab, fmt in [("MW","Molecular weight (MW)","{:.2f}"),
                        ("LogP","Partition coefficient (LogP)","{:.2f}"),
                        ("HBA","H-bond acceptors (HBA)","{:d}"),
                        ("HBD","H-bond donors (HBD)","{:d}"),
                        ("TPSA","Topological polar surface area","{:.2f}"),
                        ("Fsp3","Fraction of sp3 carbons (Fsp3)","{:.3f}"),
                        ("SA","Synthetic accessibility (SA)","{:.2f}"),
                        ("MCE18","MCE-18 score","{:d}"),
                        ("NP_Score","Natural-product likeness","{:.2f}")]:
        v = r[k]
        print(f"  {lab:<34}: " + (fmt.format(v) if v is not None else "n/a"))

    print("\nEIGHT CRITERIA")
    for key, lab in CRITERIA:
        v = r[key]
        mark = "PASS      " if v is True else ("FAIL      " if v is False else "not scored")
        print(f"  {lab:<34}: {mark}")

    print(f"\nCriteria passed : {r['Criteria_passed']}/8")
    print("VERDICT         : " + ("PASS - drug-like, eligible for molecular docking."
                                  if r["Final_pass"] else
                                  "FAIL - rejected by the medicinal chemistry filter."))

from rdkit.Chem.Draw import IPythonConsole
Chem.MolFromSmiles(query_SMILES)


### 3B. Multiple compounds (CSV)

In [ ]:
#@title Enter the path to your data file { display-mode: "form" }
#@markdown Upload a CSV to Colab (folder icon on the left) and give its path,
#@markdown or keep the default to run on the study's 1056-compound library.

data_path    = "https://raw.githubusercontent.com/phuocchung123/stat3/main/data/1056_compounds.csv"  #@param {type:"string"}
smiles_col   = "Smiles"  #@param {type:"string"}
id_col       = ""        #@param {type:"string"}
save_results = True      #@param {type:"boolean"}

data = pd.read_csv(data_path)
if id_col and id_col in data.columns:
    ids = data[id_col].astype(str)
else:
    ids = pd.Series([f"compound_{i+1}" for i in range(len(data))])

rows = []
for cid, smi in tqdm(list(zip(ids, data[smiles_col])), total=len(data), desc="Screening"):
    r = evaluate(smi)
    if r is None:
        continue
    r["ID"] = cid
    r["Smiles"] = smi
    rows.append(r)

res = pd.DataFrame(rows)
cols = ["ID", "Smiles", "MW", "LogP", "HBA", "HBD", "TPSA", "Fsp3", "SA",
        "MCE18", "NP_Score"] + [k for k, _ in CRITERIA] + ["Criteria_passed", "Final_pass"]
res = res[[c for c in cols if c in res.columns]]
passed = res[res["Final_pass"]]

print(f"\nValid compounds       : {len(res)}")
print(f"Passed the filter     : {len(passed)}")
print(f"Rejected              : {len(res) - len(passed)}\n")
print("Compounds FAILING each criterion:")
for key, lab in CRITERIA:
    if key in res.columns:
        print(f"  {lab:<34}: {int((res[key] == False).sum())}")

if save_results:
    res.to_csv("screening_results_full.csv", index=False)
    passed.to_csv("screening_results_passed.csv", index=False)
    print("\nSaved: screening_results_full.csv and screening_results_passed.csv")

passed.head(10)


In [ ]:
#@title (Optional) Download the results { display-mode: "form" }
try:
    from google.colab import files
    files.download("screening_results_passed.csv")
except Exception as e:
    print("Only works on Google Colab.", e)


## 4. Molecular docking with Smina

Docking is **not run on Colab**: it needs a prepared receptor file (`*.pdbqt`)
and substantial CPU time. The commands below are the ones used in the study,
taken from [`Smina-6puw.ipynb`](https://github.com/phuocchung123/stat3/blob/main/Smina-6puw.ipynb), to reproduce the run
on a local Linux machine.

```bash
wget https://downloads.sourceforge.net/project/smina/smina.static
chmod +x smina.static
```

```python
import glob, os

ligand_dir = "<directory_of_ligand_pdbqt_files>"
receptor   = "<path>/6njs.pdbqt"
out_dir    = "<output_directory>"
log_dir    = "<log_directory>"

ligand_name = [os.path.basename(p)[:-6]
               for p in sorted(glob.glob(f"{ligand_dir}/*.pdbqt"))]

for i in ligand_name:
    !./smina.static --receptor "{receptor}" \
                    --ligand   "{ligand_dir}/{i}.pdbqt" \
                    --center_x 12.599 --center_y 54.028 --center_z 1.242 \
                    --size_x 25 --size_y 25 --size_z 25 \
                    --cpu 24 --seed 42 --exhaustiveness 150 --num_modes 100 \
                    --out "{out_dir}/{i}.pdbqt" > "{log_dir}/{i}.txt"
```

**Binding-site box for receptor 6NJS**

| Setting | Value |
|---|---|
| Box centre (x, y, z) | 12.599 / 54.028 / 1.242 |
| Box size (x, y, z) | 25 / 25 / 25 Å |
| Random seed | 42 |
| Exhaustiveness | 150 |
| num_modes | 100 |

Activity threshold for the Smina model: **−5.989 kcal/mol**, taken from the
highest G-mean point on the ROC curve. A compound scoring below this threshold
is considered a potential STAT3 inhibitor.
